In [ ]:
def sample_population_conditional(
    samples, samples_z, objectives_z, dist_params,
    shape, location, scale, if_converged,
    pop_size, n_selected, n_obj, n_con, capacity, rng):

    pop_count = 0
    population = np.zeros((pop_size, n_selected), dtype=np.int32)
    n_items = samples.shape[0]

    while pop_count < pop_size:
        knapsack_indices = np.zeros(n_selected, dtype=int)    
        knapsack = np.zeros((n_selected,(n_obj+n_con)))

        probabilities = np.ones(n_items) / n_items  # sample first item from uniform

        # single objective (objective 1)
        # r = samples[:, 0].argsort().argsort().astype(float)
        # s = r / (r.max() + 1e-12)
        # logits = s / 0.1
        # logits -= logits.max() # avoid overflow
        # p_rank = np.exp(logits)
        # p_rank /= p_rank.sum()
        # probabilities = probabilities * p_rank
        # probabilities /= probabilities.sum()

        first_choice = rng.choice(n_items, p=probabilities)
        knapsack_indices[0] = first_choice
        knapsack[0, :] = samples[first_choice, :]
        y_prev_z = samples_z[first_choice, :] 
        y_cum = knapsack[0, :].copy()

        for n in range(1, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])   
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]
            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
            probabilities = densities / (densities.sum() + 1e-12)

            # single objective (objective 1)
            # r = x_candidates[:, 0].argsort().argsort().astype(float)
            r = x_candidates[:, 0]
            s = r / (r.max() + 1e-12)
            logits = s / 0.03
            logits -= logits.max() # avoid overflow
            p_rank = np.exp(logits)
            p_rank /= p_rank.sum()
            probabilities = probabilities * p_rank
            probabilities /= probabilities.sum() 
            
            # multi objectives (objective 1 and 2)
            # r1 = x_candidates[:, 0].argsort().argsort().astype(float)
            # r2 = x_candidates[:, 1].argsort().argsort().astype(float)
            # r_c = np.hstack((r1[:, None], r2[:, None]))
            # r = np.linalg.norm(r_c, axis=1, keepdims=True).reshape(-1)
            # s = r / (r.max() + 1e-12)
            # logits = s / 0.03
            # logits -= logits.max() # avoid overflow
            # p_rank = np.exp(logits)
            # p_rank /= p_rank.sum()
            # probabilities = probabilities * p_rank
            # probabilities /= probabilities.sum() 
            
            # aspiration point
            # aspi = np.array([60, 0, 0])
            # unit_aspi = aspi / np.linalg.norm(aspi)
            # unit_x_candidates = x_candidates[:, :n_obj] / np.linalg.norm(x_candidates[:, :n_obj], axis=1, keepdims=True)
            # weights = unit_x_candidates @ unit_aspi
            # probabilities = probabilities * weights
            # probabilities /= probabilities.sum()
            
            next_choice = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]
            
            # normalize y
            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)
        
        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population[pop_count, :] = knapsack_indices
            pop_count += 1
    
    return population

In [ ]:
def sample_population_conditional_converged(
    samples, samples_z, objectives_z, dist_params,
    shape, location, scale, if_converged,
    pop_size, n_selected, n_obj, n_con, capacity, rng):

    pop_count = 0
    population = np.zeros((pop_size, n_selected), dtype=np.int32)
    n_items = samples.shape[0]

    while pop_count < pop_size:
        knapsack_indices = np.zeros(n_selected, dtype=int)    
        knapsack = np.zeros((n_selected,(n_obj+n_con)))

        probabilities = base_rate_model(samples_z, objectives_z[:, 0, :]) # sample first item from base rate model

        # single objective (objective 1)
        r = samples[:, 0].argsort().argsort().astype(float)
        s = r / (r.max() + 1e-12)
        logits = s / 0.1
        logits -= logits.max() # avoid overflow
        p_rank = np.exp(logits)
        p_rank /= p_rank.sum()
        probabilities = probabilities * p_rank
        probabilities /= probabilities.sum()

        first_choice = rng.choice(n_items, p=probabilities)
        knapsack_indices[0] = first_choice
        knapsack[0, :] = samples[first_choice, :]
        y_prev_z = samples_z[first_choice, :] 
        y_cum = knapsack[0, :].copy()

        for n in range(1, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])   
            x_candidates_z = samples_z[x_indices, :] 
            x_candidates = samples[x_indices, :]
            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
            probabilities = densities / (densities.sum() + 1e-12)

            # single objective (objective 1)
            # r = x_candidates[:, 0].argsort().argsort().astype(float)
            r = x_candidates[:, 0]
            s = r / (r.max() + 1e-12)
            logits = s / 0.03
            logits -= logits.max()
            p_rank = np.exp(logits)
            p_rank /= p_rank.sum()
            probabilities = probabilities * p_rank
            probabilities /= probabilities.sum() 

            # multi objectives (objective 1 and 2)
            # r1 = x_candidates[:, 0].argsort().argsort().astype(float)
            # r2 = x_candidates[:, 1].argsort().argsort().astype(float)
            # r_c = np.hstack((r1[:, None], r2[:, None]))
            # r = np.linalg.norm(r_c, axis=1, keepdims=True).reshape(-1)
            # s = r / (r.max() + 1e-12)
            # logits = s / 0.03
            # logits -= logits.max() # avoid overflow
            # p_rank = np.exp(logits)
            # p_rank /= p_rank.sum()
            # probabilities = probabilities * p_rank
            # probabilities /= probabilities.sum() 

            # aspiration point
            # aspi = np.array([60, 0, 0])
            # unit_aspi = aspi / np.linalg.norm(aspi)
            # unit_x_candidates = x_candidates[:, :n_obj] / np.linalg.norm(x_candidates[:, :n_obj], axis=1, keepdims=True)
            # weights = unit_x_candidates @ unit_aspi
            # probabilities = probabilities * weights
            # probabilities /= probabilities.sum()
            
            next_choice = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]
            
            # normalize y
            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)
        
        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population[pop_count, :] = knapsack_indices
            pop_count += 1
    
    return population

In [ ]:
# heuristic-cpEDA blend
def best_ratio_item(items, obj_to_max):
    ratios = items[:, obj_to_max] / items[:, -1]
    sort_indices = np.argsort(ratios)
    best_item_index = sort_indices[-1]
    return best_item_index

def sample_population_conditional(
    samples, samples_z, objectives_z, dist_params,
    shape, location, scale, if_converged,
    pop_size, n_selected, n_obj, n_con, capacity, rng):

    pop_count = 0
    population = np.zeros((pop_size, n_selected), dtype=np.int32)
    n_items = samples.shape[0]

    while pop_count < pop_size:
        knapsack_indices = np.zeros(n_selected, dtype=int)    
        knapsack = np.zeros((n_selected,(n_obj+n_con)))

        probabilities = np.ones(n_items) / n_items  # sample first item from uniform
        first_choice = rng.choice(n_items, p=probabilities)
        knapsack_indices[0] = first_choice
        knapsack[0, :] = samples[first_choice, :]
        y_prev_z = samples_z[first_choice, :] 
        y_cum = knapsack[0, :].copy()

        for n in range(1, 3):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]
            # obj_to_max = rng.integers(n_obj)
            obj_to_max = 0
            
            next_choice = best_ratio_item(x_candidates, obj_to_max)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]

            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)
        
        for n in range(3, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])   
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]
            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
            probabilities = densities / (densities.sum())
            
            next_choice = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]
            
            # normalize y
            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)
        
        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population[pop_count, :] = knapsack_indices
            pop_count += 1
    
    return population

In [ ]:
# heuristic-cpEDA blend
def sample_population_conditional_converged(
    samples, samples_z, objectives_z, dist_params,
    shape, location, scale, if_converged,
    pop_size, n_selected, n_obj, n_con, capacity, rng):

    pop_count = 0
    population = np.zeros((pop_size, n_selected), dtype=np.int32)
    n_items = samples.shape[0]

    while pop_count < pop_size:
        knapsack_indices = np.zeros(n_selected, dtype=int)    
        knapsack = np.zeros((n_selected,(n_obj+n_con)))

        probabilities = base_rate_model(samples_z, objectives_z[:, 0, :]) # sample first item from base rate model
        first_choice = rng.choice(n_items, p=probabilities)
        knapsack_indices[0] = first_choice
        knapsack[0, :] = samples[first_choice, :]
        y_prev_z = samples_z[first_choice, :] 
        y_cum = knapsack[0, :].copy()

        for n in range(1, 3):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]
            # obj_to_max = rng.integers(n_obj)
            obj_to_max = 0
            
            next_choice = best_ratio_item(x_candidates, obj_to_max)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]

            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)
        
        for n in range(3, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])   
            x_candidates_z = samples_z[x_indices, :] 
            x_candidates = samples[x_indices, :]
            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
            probabilities = densities / (densities.sum())
            
            next_choice = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]
            
            # normalize y
            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)
        
        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population[pop_count, :] = knapsack_indices
            pop_count += 1
    
    return population

In [ ]:
# parallel processing sampling function (original)
import multiprocessing
from functools import partial
from scipy.special import gammainc, ndtri

def _sample_chunk(seed, chunk_size, samples, samples_z, base_model_params, cond_models, 
                  shape, location, scale,
                  n_selected, n_obj, n_con, capacity):
    rng = np.random.default_rng(seed)
    n_items = samples.shape[0]
    population_chunk = np.zeros((chunk_size, n_selected), dtype=np.int32)
    pop_count = 0
    total_attempts = 0
    
    while pop_count < chunk_size:
        total_attempts += 1
        
        knapsack_indices = np.zeros(n_selected, dtype=int)    
        knapsack = np.zeros((n_selected, (n_obj + n_con)))
        
        probabilities = calculate_probs_base(base_model_params, samples_z)
        first_choice = rng.choice(n_items, p=probabilities)
        knapsack_indices[0] = first_choice
        knapsack[0, :] = samples[first_choice, :]
        y_prev_z = samples_z[first_choice, :]     
        for n in range(1, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])   
            x_candidates = samples_z[x_indices, :] 
            # step n (1 to N-1) corresponds to cond_models n-1
            probs = calculate_probs_conditional(cond_models[n-1], x_candidates, y_prev_z)
            next_choice_idx = rng.choice(len(probs), p=probs)
            next_index = x_indices[next_choice_idx]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]
            
            # normalize y
            u = gamma.cdf(knapsack[n, :], a=shape[n, :], loc=location[n, :], scale=scale[n, :])
            # u = gammainc(shape[n, :], knapsack[n, :] / scale[n, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)
     
        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population_chunk[pop_count, :] = knapsack_indices
            pop_count += 1
        
        if total_attempts % 5000 == 0:
            print(f"  Worker pid={multiprocessing.current_process().pid}: "
                  f"Sampled {pop_count}/{chunk_size} (Attempts: {total_attempts})", flush=True)

                
    return population_chunk

def sample_population_conditional(
    samples, samples_z, objectives_z, dist_params, 
    shape, location, scale,
    pop_size, n_selected, n_obj, n_con, capacity, rng):

    base_model = get_base_model_params(samples_z, objectives_z[:, 0, :])
    cond_models = get_conditional_params(dist_params)
    
    n_jobs = multiprocessing.cpu_count()
    chunk_size = pop_size // n_jobs
    remainder = pop_size % n_jobs
    chunks = [chunk_size] * n_jobs
    if remainder > 0:
        chunks[-1] += remainder
        
    seeds = rng.integers(0, 1e9, size=n_jobs)
    with multiprocessing.Pool(processes=n_jobs) as pool:
        func = partial(_sample_chunk, 
                       samples=samples, 
                       samples_z=samples_z, 
                       base_model_params=base_model,
                       cond_models=cond_models,
                       shape=shape,
                       location=location,
                       scale=scale,
                       n_selected=n_selected,
                       n_obj=n_obj,
                       n_con=n_con, 
                       capacity=capacity)
        
        results = pool.starmap(func, zip(seeds, chunks))
        
    population = np.vstack(results)
    return population

In [ ]:
## parallel processing sampling function (modified to match sample_population_conditional_converged)
import multiprocessing
from functools import partial
import numpy as np
from scipy.stats import gamma, norm

def _sample_chunk(seed, chunk_size, samples, samples_z, objectives_z, dist_params,
                  shape, location, scale,
                  n_selected, n_obj, n_con, capacity):
    rng = np.random.default_rng(seed)
    n_items = samples.shape[0]
    population_chunk = np.zeros((chunk_size, n_selected), dtype=np.int32)
    pop_count = 0
    total_attempts = 0

    while pop_count < chunk_size:
        total_attempts += 1

        knapsack_indices = np.zeros(n_selected, dtype=int)

        # ===== first item: base_rate_model * rank-bias on objective 1 =====
        probabilities = base_rate_model(samples_z, objectives_z[:, 0, :]).astype(float, copy=False)

        # single objective (objective 1): rank-based bias (same as converged)
        r = samples[:, 0].argsort().argsort().astype(float)
        s = r / (r.max() + 1e-12)
        logits = s / 0.1
        logits -= logits.max()
        p_rank = np.exp(logits)
        p_rank /= (p_rank.sum() + 1e-12)

        probabilities = probabilities * p_rank
        probabilities /= (probabilities.sum() + 1e-12)

        first_choice = rng.choice(n_items, p=probabilities)
        knapsack_indices[0] = first_choice

        y_prev_z = samples_z[first_choice, :].copy()
        y_cum = samples[first_choice, :].astype(float, copy=True)

        # ===== subsequent items =====
        for n in range(1, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]

            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n).astype(float, copy=False)
            probabilities = densities / (densities.sum() + 1e-12)

            # single objective (objective 1): NOTE you used raw values (not ranks) in converged
            r = x_candidates[:, 0].astype(float, copy=False)
            s = r / (r.max() + 1e-12)
            logits = s / 0.03
            logits -= logits.max()
            p_rank = np.exp(logits)
            p_rank /= (p_rank.sum() + 1e-12)

            probabilities = probabilities * p_rank
            probabilities /= (probabilities.sum() + 1e-12)

            next_choice_idx = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice_idx]
            knapsack_indices[n] = next_index

            # update cumulative y and then normalize to z (same as converged)
            y_cum += samples[next_index, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1 - 1e-12)
            y_prev_z = norm.ppf(u)

        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population_chunk[pop_count, :] = knapsack_indices
            pop_count += 1

        if total_attempts % 5000 == 0:
            print(f"  Worker pid={multiprocessing.current_process().pid}: "
                  f"Sampled {pop_count}/{chunk_size} (Attempts: {total_attempts})", flush=True)

    return population_chunk


def sample_population_conditional_converged_parallel(
    samples, samples_z, objectives_z, dist_params,
    shape, location, scale,
    pop_size, n_selected, n_obj, n_con, capacity, rng):

    n_jobs = multiprocessing.cpu_count()
    chunk_size = pop_size // n_jobs
    remainder = pop_size % n_jobs
    chunks = [chunk_size] * n_jobs
    if remainder > 0:
        chunks[-1] += remainder

    seeds = rng.integers(0, 1e9, size=n_jobs)

    with multiprocessing.Pool(processes=n_jobs) as pool:
        func = partial(_sample_chunk,
                       samples=samples,
                       samples_z=samples_z,
                       objectives_z=objectives_z,
                       dist_params=dist_params,
                       shape=shape,
                       location=location,
                       scale=scale,
                       n_selected=n_selected,
                       n_obj=n_obj,
                       n_con=n_con,
                       capacity=capacity)

        results = pool.starmap(func, zip(seeds, chunks))

    population = np.vstack(results)
    return population


STORE CURRENT FILE:

In [ ]:
def best_ratio_item(items, obj_to_max):
    ratios = items[:, obj_to_max] / items[:, -1]
    sort_indices = np.argsort(ratios)
    best_item_index = sort_indices[-1]
    return best_item_index

In [ ]:
# parallel processing sampling functions
import multiprocessing
from functools import partial

def _sample_chunk(
    seed, chunk_size, 
    samples, samples_z, objectives_z, dist_params, 
    shape, location, scale, if_converged,
    n_selected, n_obj, n_con, capacity
):
    
    rng = np.random.default_rng(seed)
    n_items = samples.shape[0]
    population_chunk = np.zeros((chunk_size, n_selected), dtype=np.int32)
    pop_count = 0
    total_attempts = 0

    while pop_count < chunk_size:
        total_attempts += 1

        knapsack_indices = np.zeros(n_selected, dtype=int)
        knapsack = np.zeros((n_selected, (n_obj + n_con)))
        # probabilities = np.ones(n_items, dtype=float) / n_items

        # ---- multi objectives (objective 1 and 2) ----
        r1 = samples[:, 0].argsort().argsort().astype(float)
        r2 = samples[:, 1].argsort().argsort().astype(float)
        r_c = np.hstack((r1[:, None], r2[:, None]))
        r = np.linalg.norm(r_c, axis=1, keepdims=True).reshape(-1)
        s = r / (r.max() + 1e-12)
        logits = s / 0.1
        logits -= logits.max()
        p_rank = np.exp(logits)
        p_rank /= p_rank.sum()
        # probabilities = probabilities * p_rank
        # probabilities /= probabilities.sum() 
        # -------------------------------- 
        
        first_choice = rng.choice(n_items, p=p_rank)
        knapsack_indices[0] = first_choice
        knapsack[0, :] = samples[first_choice, :]
        y_prev_z = samples_z[first_choice, :]
        y_cum = knapsack[0, :].copy()

        for n in range(1, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]
            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
            probabilities = densities / (densities.sum() + 1e-12)

            # ---- multi objectives (objective 1 and 2) ----
            r1 = x_candidates[:, 0].argsort().argsort().astype(float)
            r2 = x_candidates[:, 1].argsort().argsort().astype(float)
            r_c = np.hstack((r1[:, None], r2[:, None]))
            r = np.linalg.norm(r_c, axis=1, keepdims=True).reshape(-1)
            s = r / (r.max() + 1e-12)
            logits = s / 0.03
            logits -= logits.max() # avoid overflow
            p_rank = np.exp(logits)
            p_rank /= p_rank.sum()
            probabilities = probabilities * p_rank
            probabilities /= probabilities.sum()
            # -------------------------------- 

            next_choice = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]

            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n - 1, :], loc=location[n - 1, :], scale=scale[n - 1, :])
            u = np.clip(u, 1e-12, 1-1e-12)
            y_prev_z = norm.ppf(u)

        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population_chunk[pop_count, :] = knapsack_indices
            pop_count += 1
        
        if total_attempts % 5000 == 0:
            print(f"  Worker pid={multiprocessing.current_process().pid}: "
                  f"Sampled {pop_count}/{chunk_size} (Attempts: {total_attempts})", flush=True)
                  
    return population_chunk

def _sample_chunk_converged(
    seed, chunk_size, 
    samples, samples_z, objectives_z, dist_params,
    shape, location, scale, if_converged,
    n_selected, n_obj, n_con, capacity
):
    
    rng = np.random.default_rng(seed)
    n_items = samples.shape[0]
    population_chunk = np.zeros((chunk_size, n_selected), dtype=np.int32)
    pop_count = 0
    total_attempts = 0

    while pop_count < chunk_size:
        total_attempts += 1
        knapsack_indices = np.zeros(n_selected, dtype=int)
        knapsack = np.zeros((n_selected,(n_obj+n_con)))
        # probabilities = base_rate_model(samples_z, objectives_z[:, 0, :])

        # ---- multi objectives (objective 1 and 2) ----
        r1 = samples[:, 0].argsort().argsort().astype(float)
        r2 = samples[:, 1].argsort().argsort().astype(float)
        r_c = np.hstack((r1[:, None], r2[:, None]))
        r = np.linalg.norm(r_c, axis=1, keepdims=True).reshape(-1)
        s = r / (r.max() + 1e-12)
        logits = s / 0.1
        logits -= logits.max() # avoid overflow
        p_rank = np.exp(logits)
        p_rank /= p_rank.sum()
        # probabilities = probabilities * p_rank
        # probabilities /= probabilities.sum()
        # --------------------------------

        first_choice = rng.choice(n_items, p=p_rank)
        knapsack_indices[0] = first_choice
        knapsack[0, :] = samples[first_choice, :]
        y_prev_z = samples_z[first_choice, :] 
        y_cum = knapsack[0, :].copy()

        for n in range(1, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]
            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
            probabilities = densities / (densities.sum() + 1e-12)

            # ---- multi objectives (objective 1 and 2) ----
            r1 = x_candidates[:, 0].argsort().argsort().astype(float)
            r2 = x_candidates[:, 1].argsort().argsort().astype(float)
            r_c = np.hstack((r1[:, None], r2[:, None]))
            r = np.linalg.norm(r_c, axis=1, keepdims=True).reshape(-1)
            s = r / (r.max() + 1e-12)
            logits = s / 0.03
            logits -= logits.max() # avoid overflow
            p_rank = np.exp(logits)
            p_rank /= p_rank.sum()
            probabilities = probabilities * p_rank
            probabilities /= probabilities.sum()
            # --------------------------------

            next_choice = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]

            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1 - 1e-12)
            y_prev_z = norm.ppf(u)

        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population_chunk[pop_count, :] = knapsack_indices
            pop_count += 1

        if total_attempts % 5000 == 0:
            print(f"  Worker pid={multiprocessing.current_process().pid}: "
                  f"Sampled {pop_count}/{chunk_size} (Attempts: {total_attempts})", flush=True)

    return population_chunk

def _sample_chunk_blend(
    seed, chunk_size, 
    samples, samples_z, objectives_z, dist_params, 
    shape, location, scale, if_converged,
    n_selected, n_obj, n_con, capacity
):

    rng = np.random.default_rng(seed)
    n_items = samples.shape[0]
    population_chunk = np.zeros((chunk_size, n_selected), dtype=np.int32)
    pop_count = 0
    total_attempts = 0
    
    while pop_count < chunk_size:
        total_attempts += 1
        
        knapsack_indices = np.zeros(n_selected, dtype=int)    
        knapsack = np.zeros((n_selected, (n_obj + n_con)))
        y_cum = np.zeros((n_obj + n_con))
        for n in range(0, 3):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])
            x_candidates = samples[x_indices, :]
            obj_to_max = 0  

            next_choice_local = best_ratio_item(x_candidates, obj_to_max)
            next_index = x_indices[next_choice_local]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]

            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1 - 1e-12)
            y_prev_z = norm.ppf(u)

        for n in range(3, n_selected):
            x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])   
            x_candidates_z = samples_z[x_indices, :]
            x_candidates = samples[x_indices, :]
            densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
            probabilities = densities / (densities.sum())
            
            next_choice = rng.choice(len(probabilities), p=probabilities)
            next_index = x_indices[next_choice]
            knapsack_indices[n] = next_index
            knapsack[n, :] = samples[next_index, :]

            y_cum += knapsack[n, :]
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
            u = np.clip(u, 1e-12, 1 - 1e-12)
            y_prev_z = norm.ppf(u)

        constraint = np.sum(samples[knapsack_indices, -1])
        if constraint <= capacity:
            population_chunk[pop_count, :] = knapsack_indices
            pop_count += 1
        
        if total_attempts % 5000 == 0:
            print(f"  Worker pid={multiprocessing.current_process().pid}: "
                  f"Sampled {pop_count}/{chunk_size} (Attempts: {total_attempts})", flush=True)

    return population_chunk


def sample_population_conditional_parallel(
    samples, samples_z, objectives_z, dist_params, 
    shape, location, scale, if_converged, 
    pop_size, n_selected, n_obj, n_con, capacity, rng, 
    n_jobs=None
):

    if n_jobs is None:
        n_jobs = multiprocessing.cpu_count()
        
    base = pop_size // n_jobs
    rem = pop_size % n_jobs
    chunks = [base] * n_jobs
    for i in range(rem):
        chunks[i] += 1
    chunks = [c for c in chunks if c > 0]
    n_jobs = len(chunks)

    seeds = rng.integers(0, 1_000_000_000, size=n_jobs, dtype=np.int64)

    with multiprocessing.Pool(processes=n_jobs) as pool:
        func = partial(
            _sample_chunk,
            samples=samples,
            samples_z=samples_z,
            objectives_z=objectives_z,
            dist_params=dist_params,
            shape=shape,
            location=location,
            scale=scale,
            if_converged=if_converged,
            n_selected=n_selected,
            n_obj=n_obj,
            n_con=n_con,
            capacity=capacity,
        )
        results = pool.starmap(func, zip(seeds.tolist(), chunks))

    population = np.vstack(results)
    return population

def sample_population_conditional_parallel_converged(
    samples, samples_z, objectives_z, dist_params, 
    shape, location, scale, if_converged, 
    pop_size, n_selected, n_obj, n_con, capacity, rng, 
    n_jobs=None
):

    if n_jobs is None:
        n_jobs = multiprocessing.cpu_count()
        
    base = pop_size // n_jobs
    rem = pop_size % n_jobs
    chunks = [base] * n_jobs
    for i in range(rem):
        chunks[i] += 1
    chunks = [c for c in chunks if c > 0]
    n_jobs = len(chunks)

    seeds = rng.integers(0, 1_000_000_000, size=n_jobs, dtype=np.int64)

    with multiprocessing.Pool(processes=n_jobs) as pool:
        func = partial(
            _sample_chunk_converged,
            samples=samples,
            samples_z=samples_z,
            objectives_z=objectives_z,
            dist_params=dist_params,
            shape=shape,
            location=location,
            scale=scale,
            if_converged=if_converged,
            n_selected=n_selected,
            n_obj=n_obj,
            n_con=n_con,
            capacity=capacity,
        )
        results = pool.starmap(func, zip(seeds.tolist(), chunks))

    population = np.vstack(results)
    return population

In [ ]:
class KnapsackEDACond:
    def __init__(self, items, capacity, n_selected, n_obj, n_con, pop_size=1000, 
                 generations=10, max_no_improve_gen=10, max_iters=100, seed=1123):
        self.items = items
        self.capacity = capacity
        self.n_selected = n_selected
        self.n_obj = n_obj
        self.n_con = n_con
        self.pop_size = pop_size
        self.generations = generations
        self.max_no_improve_gen = max_no_improve_gen
        self.max_iters = max_iters
        self.rng = random.default_rng(seed=seed)

        self.items_z = None
        # self.ecdf_table = None
        self.shape = None
        self.location = None
        self.scale = None
        self.if_initial = True
        self.if_converged = False

        self.first_item_dist = None
        self.distribution_params = None
        self.selected_population = None  # (pop_size, n_selected)
        self.selected_objectives = None  # (pop_size, n_obj) objective values are summed over solutions
        self.objectives_z = None  # (pop_size, n_selected, n_obj)

        self.distribution_params_table = []
        self.pareto_indices_table = []
        self.pareto_front_table = []
        self.js_div_list = []
        self.converged_pf_table = []

    def _generate_initial_population(self):
        n_items = self.items.shape[0]
        distribution = np.ones(n_items) / n_items
        population = sample_population(
            self.items, distribution, self.pop_size, self.n_selected, 
            self.capacity, self.rng
        )
        objectives = get_objectives(self.items, population, self.n_obj)
        
        ranks, fronts = non_dominated_sort(objectives)
        distances_all_solutions = np.zeros(population.shape[0], dtype=float)
        for f in fronts:
            distances = assign_crowding_distance(objectives[f, :])
            distances_all_solutions[f] = distances

        select_indices = np.array([], dtype=int)
        while len(select_indices) < self.pop_size:
            indice = binary_tournament_selection(
                population, ranks, distances_all_solutions, self.rng
            )
            select_indices = np.concatenate([select_indices, np.array([indice])])
        
        selected_population = population[select_indices]
        selected_objectives = objectives[select_indices]

        _, _, self.items_z = transform_items_to_z(self.items)
        self.objectives_z, self.distribution_params, self.shape, self.location, self.scale = fit_conditional(self.items, self.items_z, selected_population, 
                                                                                                                self.n_selected, self.n_obj, self.n_con,
                                                                                                                self.rng, self.if_initial) # may need a different rng
        self.first_item_dist = base_rate_model(self.items_z, self.objectives_z[:, 0, :])
        self.selected_population = selected_population
        self.selected_objectives = selected_objectives

    
    def _update_distribution(self):
        # sampling
        population = sample_population_conditional_parallel(
            self.items, self.items_z, self.objectives_z, self.distribution_params,
            self.shape, self.location, self.scale, self.if_converged,
            self.pop_size, self.n_selected, self.n_obj, self.n_con, self.capacity, self.rng
        )
        objectives = get_objectives(self.items, population, self.n_obj)
        
        # find current pareto front
        _, fronts_current = non_dominated_sort(objectives)
        pareto_indices = population[fronts_current[0]]
        
        # stack populations
        objectives = np.vstack((self.selected_objectives, objectives))
        population = np.vstack((self.selected_population, population))
        
        # select through non-dominated sorting
        ranks, fronts = non_dominated_sort(objectives)
        select_indices = np.array([], dtype=np.int32)
        for f in fronts:
            if len(select_indices) + len(f) <= self.pop_size:
                select_indices = np.concatenate([select_indices, f])
            else:
                remaining_size = self.pop_size - len(select_indices)
                f_distance = assign_crowding_distance(objectives[f, :])
                sort_indices = np.argsort(f_distance)[::-1]
                remaining = f[sort_indices[:remaining_size]]
                select_indices = np.concatenate([select_indices, remaining])
                break
        selected_population = population[select_indices]
        selected_objectives = objectives[select_indices]
        
        # determine training population
        # if self.if_converged:
        #     training_population = population[fronts[0]]
        # else:
        n_training = int(self.pop_size*0.15)
        training_population = selected_population[:n_training]
        
        # update distribution
        self.objectives_z, self.distribution_params, self.shape, self.location, self.scale= fit_conditional(self.items, self.items_z, training_population, 
                                                                                                                self.n_selected, self.n_obj, self.n_con,
                                                                                                                self.rng, self.if_initial)
        
        self.selected_population = selected_population
        self.selected_objectives = selected_objectives

        # compute js divergence
        updated_first_item_dist = base_rate_model(self.items_z, self.objectives_z[:, 0, :])
        self.first_item_dist[self.first_item_dist < 1E-08] = 1E-08
        updated_first_item_dist[updated_first_item_dist < 1E-08] = 1E-08
        js_div = jensenshannon(self.first_item_dist, updated_first_item_dist)**2
        self.first_item_dist = updated_first_item_dist
        
        return pareto_indices, js_div

    def _converged_pf(self):
        # sampling
        population = sample_population_conditional_parallel_converged(
            self.items, self.items_z, self.objectives_z, self.distribution_params,
            self.shape, self.location, self.scale, self.if_converged,
            self.pop_size, self.n_selected, self.n_obj, self.n_con, self.capacity, self.rng)
        objectives = get_objectives(self.items, population, self.n_obj)

        # find current pareto front
        pareto_indices = population[non_dominated(objectives).astype(bool)]
        
        # stack populations
        population = np.unique(np.sort(np.vstack((self.selected_population, population)), axis=1), axis=0)
        objectives = get_objectives(self.items, population, self.n_obj)

        # select through non-dominated
        nd_idx = non_dominated(objectives).astype(bool)
        selected_population = population[nd_idx]
        selected_objectives = objectives[nd_idx]

        # update distribution
        self.objectives_z, self.distribution_params, self.shape, self.location, self.scale = fit_conditional(self.items, self.items_z, selected_population, 
                                                                                                                self.n_selected, self.n_obj, self.n_con,
                                                                                                                self.rng, self.if_initial)
        self.selected_population = selected_population
        self.selected_objectives = selected_objectives    

        # compute js divergence
        updated_first_item_dist = base_rate_model(self.items_z, self.objectives_z[:, 0, :])
        self.first_item_dist[self.first_item_dist < 1E-08] = 1E-08
        updated_first_item_dist[updated_first_item_dist < 1E-08] = 1E-08
        js_div = jensenshannon(self.first_item_dist, updated_first_item_dist)**2
        self.first_item_dist = updated_first_item_dist                                                                         
        
        return pareto_indices, js_div

    def run(self):
        self._generate_initial_population()
        self.if_initial = False
        
        # Run generations (fixed number of generations)
        # for g in range(self.generations):
        #     print(f"Generation {g+1}/{self.generations}")
        #     pareto_indices, js_div = self._update_distribution()
        #     print(f"number of front 0: {pareto_indices.shape[0]}")
            
        #     pareto_front = np.zeros((pareto_indices.shape[0], self.items.shape[1]))
        #     for k in range(pareto_indices.shape[0]):
        #         pareto_front[k, :] = np.sum(self.items[pareto_indices[k, :], :], axis=0)
                
        #     self.distribution_params_table.append(self.distribution_params.copy())
        #     self.pareto_indices_table.append(pareto_indices.copy())
        #     self.pareto_front_table.append(pareto_front.copy())
        #     self.js_div_list.append(js_div)

        # return {
        #     'distribution_params_table': self.distribution_params_table,
        #     'pareto_indices_table': self.pareto_indices_table,
        #     'pareto_front_table': self.pareto_front_table,
        #     'js_div_list': self.js_div_list,
        #     'objectives_z': self.objectives_z,
        #     'items_z': self.items_z,
        #     'shape': self.shape,
        #     'location': self.location,
        #     'scale': self.scale
        # }
        
        # # Run generations (until convergence)
        # part 1: train on a portion of selected population till base rate converges
        no_improve_gen = 0
        prev_js_div = None
        generation = 0
        min_gens = 30
        while no_improve_gen < self.max_no_improve_gen:
        # for generation in range(self.generations):
            generation += 1
            print(f"Generation {generation} (no improve count: {no_improve_gen})")
            pareto_indices, js_div = self._update_distribution()
            print(f"number of front 0: {pareto_indices.shape[0]}")

            pareto_front = np.zeros((pareto_indices.shape[0], self.items.shape[1]))
            for k in range(pareto_indices.shape[0]):
                pareto_front[k, :] = np.sum(self.items[pareto_indices[k, :], :], axis=0)
                
            self.distribution_params_table.append(self.distribution_params.copy())
            self.pareto_indices_table.append(pareto_indices.copy())
            self.pareto_front_table.append(pareto_front.copy())
            self.js_div_list.append(js_div)
                
            # option 1
            if prev_js_div is not None:
                diff = prev_js_div - js_div
                # if generation > min_gens and np.abs(diff) < 0.005: # criteria may change
                if np.abs(diff) < 0.005:
                    no_improve_gen += 1
                else:
                    no_improve_gen = 0
            else:
                no_improve_gen = 0
            ## option2
            # if js_div < 0.005:
            #     no_improve_gen += 1
            # else:
            #     no_improve_gen = 0
            prev_js_div = js_div
        
        # part 2: train on only the non-dominated solutions till pareto front converges
        no_improve_gen = 0
        counter = 0
        prev_front_0 = None
        while no_improve_gen < self.max_no_improve_gen and counter < self.max_iters:
            counter += 1
            print(f"Iterations {counter} (no improve count: {no_improve_gen})")
            pareto_indices, js_div = self._converged_pf()
            print(f"number of front 0: {pareto_indices.shape[0]}")

            pareto_front = np.zeros((pareto_indices.shape[0], self.items.shape[1]))
            for k in range(pareto_indices.shape[0]):
                pareto_front[k, :] = np.sum(self.items[pareto_indices[k, :], :], axis=0)

            self.distribution_params_table.append(self.distribution_params.copy())
            self.pareto_indices_table.append(pareto_indices.copy())
            self.pareto_front_table.append(pareto_front.copy())
            self.js_div_list.append(js_div)
            
            front_0 = np.unique(self.selected_objectives, axis=0)
            front_0 = front_0[np.lexsort(front_0.T[::-1])]
            if prev_front_0 is not None:
                if np.array_equal(prev_front_0, front_0):
                    no_improve_gen += 1
                else:
                    no_improve_gen = 0
            else:
                no_improve_gen = 0
            
            self.converged_pf_table.append(front_0.copy())
            prev_front_0 = front_0


        return {
            'distribution_params_table': self.distribution_params_table,
            'pareto_indices_table': self.pareto_indices_table,
            'pareto_front_table': self.pareto_front_table,
            'converged_pf_table': self.converged_pf_table,
            'js_div_list': self.js_div_list,
            'objectives_z': self.objectives_z,
            'items_z': self.items_z,
            'shape': self.shape,
            'location': self.location,
            'scale': self.scale,
            'base_rate_dist': self.first_item_dist
        }

In [ ]:
while pop_count < chunk_size:
    total_attempts += 1
    
    knapsack_indices = np.zeros(n_selected, dtype=int)    
    knapsack = np.zeros((n_selected, (n_obj + n_con)))
    y_cum = np.zeros((n_obj + n_con))
    for n in range(0, 1):
        x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])
        x_candidates = samples[x_indices, :]
        obj_to_max = 0  

        next_choice_local = best_ratio_item(x_candidates, obj_to_max)
        next_index = x_indices[next_choice_local]
        knapsack_indices[n] = next_index
        knapsack[n, :] = samples[next_index, :]

        y_cum += knapsack[n, :]
        if n == 0:
            y_prev_z = samples_z[next_index, :]
        else:
            u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :]) # bug at n = 0
            u = np.clip(u, 1e-12, 1 - 1e-12)
            y_prev_z = norm.ppf(u)

    for n in range(1, n_selected):
        x_indices = np.setdiff1d(np.arange(n_items), knapsack_indices[:n])   
        x_candidates_z = samples_z[x_indices, :]
        x_candidates = samples[x_indices, :]
        densities = conditional_density_given_Y_and_t(x_candidates_z, y_prev_z, dist_params, n)
        probabilities = densities / (densities.sum())
        
        next_choice = rng.choice(len(probabilities), p=probabilities)
        next_index = x_indices[next_choice]
        knapsack_indices[n] = next_index
        knapsack[n, :] = samples[next_index, :]

        y_cum += knapsack[n, :]
        u = gamma.cdf(y_cum, a=shape[n-1, :], loc=location[n-1, :], scale=scale[n-1, :])
        u = np.clip(u, 1e-12, 1 - 1e-12)
        y_prev_z = norm.ppf(u)

    constraint = np.sum(samples[knapsack_indices, -1])
    if constraint <= capacity:
        population_chunk[pop_count, :] = knapsack_indices
        pop_count += 1